# German Credit Scorecard

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from scorecard.data import load_german_credit

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
# Pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
# Numpy
np.set_printoptions(suppress=True, precision=4)
# Matplotlib and Seaborn
plt.rcParamsDefault["figure.figsize"] = (10, 6)
plt.rcParamsDefault["figure.dpi"] = 100
plt.rcParamsDefault["font.size"] = 11
plt.rcParamsDefault["axes.grid"] = True
plt.rcParamsDefault["grid.alpha"] = 0.3
sns.set_style("whitegrid")
sns.set_palette("muted")

SEED = 42
rng = np.random.default_rng(SEED)

In [ ]:
df = load_german_credit("../data/raw/german_credit/german.data")
df.head()

In [ ]:
df = pd.read_csv("../data/raw/german_credit/german.data", sep=" ", header=None)

In [ ]:
df.head()

In [ ]:
columns = [
    "checking_status",
    "duration",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_status",
    "employment",
    "installment_rate",
    "personal_status_sex",
    "other_parties",
    "residence_since",
    "property_magnitude",
    "age",
    "other_payment_plans",
    "housing",
    "existing_credits",
    "job",
    "num_dependents",
    "own_telephone",
    "foreign_worker",
    "target",
]
df.columns = columns

In [ ]:
df.head()

In [ ]:
df["target"] = df["target"].map({1: 0, 2: 1})  # 1 - good, 2 - bad

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.nunique()

In [ ]:
df.duplicated().sum()

In [ ]:
df["target"].value_counts(normalize=True)

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include="str").columns.tolist()
numeric_cols.remove("target")
print(f"Numeric columns: {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")

In [ ]:
df[numeric_cols].nunique().sort_values()

In [ ]:
low_card_numeric = [
    "num_dependents",
    "installment_rate",
    "residence_since",
    "existing_credits",
]
continous_numeric = [c for c in numeric_cols if c not in low_card_numeric]

### Plot:

In [ ]:
for col in numeric_cols:
    fig, ax = plt.subplots(1, 2, figsize=(12, 3))
    df[col].hist(bins=30, ax=ax[0])
    ax[0].set_title(f"{col} — histogram")
    df.boxplot(column=col, ax=ax[1])
    ax[1].set_title(f"{col} — boxplot")
    plt.show()

In [ ]:
for col in categorical_cols:
    print(df[col].value_counts())
    print()
    df[col].value_counts().plot(kind="bar", title=col)
    plt.show()

In [ ]:
for col in categorical_cols:
    rates = df.groupby(col)["target"].agg(["mean", "count"])
    rates.columns = ["default_rate", "count"]
    print(f"\n--- {col} ---")
    print(rates.sort_values("default_rate"))

In [ ]:
for col in continous_numeric:
    df[f"{col}_qbin"] = pd.qcut(df[col], q=5, duplicates="drop")
    print(df.groupby(f"{col}_qbin")["target"].mean())
    df.drop(columns=f"{col}_qbin", inplace=True)

**choose 5 features:** *checking_status, duration, savings_status, age, credit_history*

categorical: 
- checking_status
- savings_status
- credit_history

numerical:
- duration
- age

Train/test split

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

In [ ]:
print(X_train["checking_status"].value_counts())
print(X_train["credit_history"].value_counts())
print(X_train["savings_status"].value_counts())

In [ ]:
categorical_binned = ["checking_status", "savings_status", "credit_history"]

In [ ]:
X_train["duration"].value_counts().sort_index()

In [ ]:
X_train["duration_bin"], duration_edges = pd.qcut(
    X_train["duration"], q=5, retbins=True, duplicates="drop"
)
X_train["duration_bin"].value_counts()  # confirm bins are reasonably populated

In [ ]:
X_train["age_bin"], age_edges = pd.qcut(
    X_train["age"], q=5, retbins=True, duplicates="drop"
)
X_train["age_bin"].value_counts()

In [ ]:
bin_artifacts = {
    "duration": duration_edges,
    "age": age_edges,
}

In [ ]:
binned_features = [
    "checking_status",
    "savings_status",
    "credit_history",
    "duration_bin",
    "age_bin",
]

for col in binned_features:
    print(f"\n--- {col} ---")
    print(
        pd.DataFrame({col: X_train[col], "target": y_train})
        .groupby(col)["target"]
        .agg(["mean", "count"])
    )

### WOE IV

In [ ]:
def compute_woe_iv(df_binned_col, target):
    grouped = pd.DataFrame({"bin": df_binned_col, "target": target})
    summary = grouped.groupby("bin")["target"].agg(["count", "sum"])
    summary.columns = ["total", "bad"]
    summary["good"] = summary["total"] - summary["bad"]

    total_good = summary["good"].sum()
    total_bad = summary["bad"].sum()

    summary["pct_good"] = (summary["good"] + 0.5) / (total_good + 0.5)
    summary["pct_bad"] = (summary["bad"] + 0.5) / (total_bad + 0.5)

    summary["woe"] = np.log(summary["pct_good"] / summary["pct_bad"])
    summary["iv_contribution"] = (summary["pct_good"] - summary["pct_bad"]) * summary[
        "woe"
    ]

    total_iv = summary["iv_contribution"].sum()
    return summary, total_iv

In [ ]:
iv_results = {}
woe_tables = {}

for col in binned_features:
    summary, iv = compute_woe_iv(X_train[col], y_train)
    woe_tables[col] = summary
    iv_results[col] = iv

pd.Series(iv_results).sort_values(ascending=False)

### Step 12 apply same bins/woe mappings to X_test

In [ ]:
# categorical — direct WOE lookup, same mapping learned from train
for col in ["checking_status", "savings_status", "credit_history"]:
    woe_map = woe_tables[col]["woe"].to_dict()
    X_test[f"{col}_woe"] = X_test[col].map(woe_map)

In [ ]:
X_test["duration_bin"] = pd.cut(
    X_test["duration"], bins=duration_edges, include_lowest=True
)
X_test["age_bin"] = pd.cut(X_test["age"], bins=age_edges, include_lowest=True)

X_test["duration_bin_woe"] = X_test["duration_bin"].map(
    woe_tables["duration_bin"]["woe"].to_dict()
)
X_test["age_bin_woe"] = X_test["age_bin"].map(woe_tables["age_bin"]["woe"].to_dict())

In [ ]:
woe_cols = [
    "checking_status",
    "savings_status",
    "credit_history",
    "duration_bin_woe",
    "age_bin_woe",
]

# for the categorical ones, the mapped column names would be like "checking_status_woe" etc.
# adjust names to match whatever you actually assigned
X_test[
    [
        "checking_status_woe",
        "savings_status_woe",
        "credit_history_woe",
        "duration_bin_woe",
        "age_bin_woe",
    ]
].isna().sum()

In [ ]:
# find the actual row(s) with the NaN
print(X_test[X_test["duration_bin_woe"].isna()][["duration", "duration_bin"]])

In [ ]:
# 1. Extend the edges once, right after you got them from qcut
duration_edges_extended = duration_edges.copy()
duration_edges_extended[0] = -np.inf
duration_edges_extended[-1] = np.inf

# 2. Re-cut X_train using the EXTENDED edges (overwrite the original duration_bin)
X_train["duration_bin"] = pd.cut(
    X_train["duration"], bins=duration_edges_extended, include_lowest=True
)

# 3. Recompute WOE/IV on this re-cut column
woe_tables["duration_bin"], iv_results["duration_bin"] = compute_woe_iv(
    X_train["duration_bin"], y_train
)

# 4. Now cut X_test with the SAME extended edges — bins will match exactly
X_test["duration_bin"] = pd.cut(
    X_test["duration"], bins=duration_edges_extended, include_lowest=True
)
X_test["duration_bin_woe"] = X_test["duration_bin"].map(
    woe_tables["duration_bin"]["woe"].to_dict()
)

X_test["duration_bin_woe"].isna().sum()  # should be 0 now

In [ ]:
X_train["checking_status_woe"] = X_train["checking_status"].map(
    woe_tables["checking_status"]["woe"].to_dict()
)
X_train["savings_status_woe"] = X_train["savings_status"].map(
    woe_tables["savings_status"]["woe"].to_dict()
)
X_train["credit_history_woe"] = X_train["credit_history"].map(
    woe_tables["credit_history"]["woe"].to_dict()
)
X_train["duration_bin_woe"] = X_train["duration_bin"].map(
    woe_tables["duration_bin"]["woe"].to_dict()
)
X_train["age_bin_woe"] = X_train["age_bin"].map(woe_tables["age_bin"]["woe"].to_dict())

# sanity check — should all be 0, since these are the bins WOE was computed FROM
X_train[
    [
        "checking_status_woe",
        "savings_status_woe",
        "credit_history_woe",
        "duration_bin_woe",
        "age_bin_woe",
    ]
].isna().sum()

### Fit the model

In [ ]:
woe_feature_cols = [
    "checking_status_woe",
    "savings_status_woe",
    "credit_history_woe",
    "duration_bin_woe",
    "age_bin_woe",
]

X_train_woe = X_train[woe_feature_cols]
X_test_woe = X_test[woe_feature_cols]

model = LogisticRegression(random_state=SEED)
model.fit(X_train_woe, y_train)

### Evaluate

In [ ]:
p_test = model.predict_proba(X_test_woe)[:, 1]
auc = roc_auc_score(y_test, p_test)
gini = 2 * auc - 1
print(f"AUC: {auc:.4f}, Gini: {gini:.4f}")

### Sanity-check coefficients

In [ ]:
pd.Series(model.coef_[0], index=woe_feature_cols).sort_values(ascending=False)

In [ ]:
from scorecard.binning import bin_continuous
from scorecard.woe import compute_woe_iv

train_binned, test_binned, edges = bin_continuous(
    X_train["duration"], X_test["duration"]
)
summary, iv = compute_woe_iv(train_binned, y_train)
print(iv)  # should match your earlier 0.2145 for duration_bin